<a href="https://colab.research.google.com/github/KateuleOrion/lab-4-llm-decision-support/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Enter Name]
**Student ID:** [Enter ID]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [2]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
# def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
#             temperature=0.7, max_tokens=500):
#     response = client.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user",   "content": user_prompt},
#         ],
#         temperature=temperature,
#         max_tokens=max_tokens,
#     )
#     return response.choices[0].message.content



def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500, return_usage=False):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    if return_usage:
        return response.choices[0].message.content, response.usage
    return response.choices[0].message.content

# TODO: Call it once and print answer and token usage
prompt = "What is the capital of France?"
answer, usage = ask_llm(prompt, return_usage=True)

print("LLM Answer")
print(answer)

print("\n Token Usage")
print(usage)

LLM Answer
The capital of France is Paris.

 Token Usage
CompletionUsage(completion_tokens=8, prompt_tokens=48, total_tokens=56, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.172773856, prompt_time=0.004469582, completion_time=0.009184877, total_time=0.013654459)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**

1. The system roles define the rules and constraint for the behaviour of large language models before begining the conversation with the user. for example,
"You are a local language tutor for beginners. Always respond in local language, followed by an English translation in parentheses. Keep the wording simple."

while User roles are the actual input passed on by the end user during an interaction for example, "How do I ask where the bathroom is?"

2. A token is the fundamental building block of text processed by an LLM.
API providers bill per token rather than per request because generating a large number of words takes a massive amount of electricity and hardware work compared to shorter ones.

### Part 1.2 — Temperature: the randomness dial

In [3]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.

question = "Suggest a name for a savings product for market traders in Accra."

# TODO: Run 5 times at temperature = 0.0

print("RESULTS AT TEMPERATURE = 0.0")

for i in range(1, 6):
    answer = ask_llm(question, temperature=0.0)
    print(f"\n Run {i}")
    print(answer)

# TODO: Run 5 times at temperature = 1.2

print("RESULTS AT TEMPERATURE = 1.2")


for i in range(1, 6):
    answer = ask_llm(question, temperature=1.2)
    print(f"\n--- Run {i} ---")
    print(answer)

RESULTS AT TEMPERATURE = 0.0

 Run 1
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Souza**: "Sika" means "money" in the Akan language, which is widely spoken in Ghana. "Souza" means "save" or "store", so this name combines a local language with a clear savings message.
4. **Market Mobi**: This name incorporates "mobi", short for mobile, to suggest a convenient and accessible savings product.
5. **Adanfo Savings**: "Adanfo" means "friends" or "partners" in the Akan language, implying a sense of community and mutual support among market traders.
6. **Kokroko Savings**: "Kokroko" means "honest" or "trustworthy" in the Akan language, conveying a sense of reliability and security.
7. **Traders' Trust**: This name emphasizes the idea of a trusted and 

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:**

At temperature = 0.0: The outputs were very consistent in tone, length, and themes (e.g., repeating words like "Makola", "Traders' Treasure", "Sika"). However, slight variations occurred across runs due to underlying hardware-level floating-point non-determinism in GPU inference.

At temperature = 1.2: The outputs showed high variability, creative rephrasing, and unpredictable structure across runs.

A low temperature of 0.0 or slightly higher is appropriate because Loan decisions require strict accuracy, consistency, and fairness. A low temperature ensures the model gives predictable, data-driven answers without introducing randomness or hallucinations into financial evaluations.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [13]:
def summarize_v1(letter_text):
    user_prompt = f"Summarize this:\n\n{letter_text}"
    return ask_llm(user_prompt=user_prompt, temperature=0.7)

# SUMMARY_PROMPT_V2 — Production Template (Role + Constraints)
SUMMARY_SYSTEM_PROMPT = """You are an expert credit analyst assistant for a microfinance institution in Ghana.
Your job is to summarize loan application letters into concise, factual briefs for a busy loan officer.

CONSTRAINTS:
1. Output MUST be exactly 3 to 4 sentences long.
2. Be objective, neutral, and strictly factual.
3. Do NOT invent, assume, or hallucinate any details not present in the text.
4. Highlight key facts: applicant identity, amount requested, stated purpose, financial status, and repayment/collateral terms."""

def summarize_v2(letter_text):
    user_prompt = f"Summarize this loan application letter:\n\n{letter_text}"
    return ask_llm(
        user_prompt=user_prompt,
        system_prompt=SUMMARY_SYSTEM_PROMPT,
        temperature=0.0
    )
test_letters = ["L002", "L006"]

for letter_id in test_letters:
    letter = LETTERS[letter_id]

    print(f"LETTER ID: {letter_id}")


    v1_out = summarize_v1(letter)
    v2_out = summarize_v2(letter)

    print("\n[V1 Output - Naive Prompt]:")
    print(v1_out)

    print("\n[V2 Output - Engineered System Prompt (Temp=0.0)]:")
    print(v2_out)
    print("\n")

# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

LETTER ID: L002

[V1 Output - Naive Prompt]:
Kwame Boateng, a commercial driver in Kumasi, is seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but is optimistic it will improve after the festive season, and he's willing to repay the loan as soon as possible. However, he doesn't have any collateral to offer at the moment.

[V2 Output - Engineered System Prompt (Temp=0.0)]:
Kwame Boateng, a commercial driver in Kumasi, is applying for a loan of GHS 25,000. The purpose of the loan is to repair his trotro engine and settle personal debts. He mentions that business has been slow, but expects it to improve after the festive season, and is willing to repay the loan when he can, although he does not have collateral to offer. The applicant is seeking urgent assistance with the loan.


LETTER ID: L006

[V1 Output - Naive Prompt]:
Kofi, a 22-year-old, is requesting GHS 50,000 to start three businesses: a car washing s

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**

1. V1 produced summaries with inconsistent lengths, whereas V2 strictly produced the mandated 3-to-4 sentence limit across both letters.

Informal Tone and Hallucinated Assumptions: V1 introduced conversational phrasing and unverified claims which v2 corrected.

for example: letter ID L006
V1: Claims "He has no prior experience..." but the letter says he hasn't started them yet, and never explicitly states his background experience.

V2: Corrects this to a strictly factual statement: "...has no prior experience in these businesses, and proposes to repay the loan in one year."

V1 omitted local contextual details that V2 preserved.

V1: Refers to a generic "vehicle's engine." while,

V2: Preserves the specific detail "trotro engine," which is inportant in context for a Ghanaian commercial transit driver.


2. Importance of "No Invented Details" Instruction
 Credit evaluation depends entirely on verifiable facts. Inventing details such as making up an applicant's income, assuming collateral exists, or inventing business experience leads to flawed risk assessments, causing financial loss through bad loans or regulatory compliance violations.

In LLM literature, this failure mode is called hallucination specifically extrapolative or unfaithful hallucination.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [15]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.




import json
import pandas as pd

# EXTRACT_PROMPT (Written with student-level phrasing)

EXTRACT_PROMPT = """You are a helpful data extraction bot for a microfinance bank in Ghana.
I need you to pull out specific details from loan letters and give me back ONLY a clean JSON object.

Here are the exact fields I need:
- "applicant_name": String with the person's full name.
- "amount_ghs": Number for how much money they want in GHS.
- "purpose": String explaining what they need the money for.
- "monthly_profit_ghs": Number showing their monthly profit in GHS (use null if they didn't mention it).
- "has_collateral_or_guarantor": Boolean (true if they offered collateral or a guarantor, false if they didn't).
- "repayment_months": Number of months they plan to pay it back (use null if not mentioned).

Rules:
1. Return ONLY the JSON object. Don't add any extra text, code blocks like ```json, or explanations.
2. If a detail isn't clearly stated in the letter, set that field to null. Don't make anything up!

Here is an example:

Input:
"Hi, my name is Chansa Bwalya. I run a shop in Accra. I am requesting a GHS 4,000 loan to buy more stock for my shop. My profit every month is about GHS 800. My brother will be my guarantor. I can finish paying back in 7 months."

Output:
{
  "applicant_name": "Chansa Bwalya",
  "amount_ghs": 4000,
  "purpose": "buy stock for shop",
  "monthly_profit_ghs": 800,
  "has_collateral_or_guarantor": true,
  "repayment_months": 7
}"""


# EXTRACTION HELPER FUNCTION
def extract_fields(letter_text):
    user_prompt = f"Extract the required data from this loan letter:\n\n{letter_text}"

    raw_response = ask_llm(
        user_prompt=user_prompt,
        system_prompt=EXTRACT_PROMPT,
        temperature=0.0
    )

    # Strip markdown backticks if the model still includes them
    cleaned = raw_response.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()

    # Parse into Python dictionary
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"[WARNING] Could not parse JSON: {e}")
        print(f"Raw response was:\n{raw_response}")
        return None

# RUN EXTRACTION ON ALL 6 LETTERS AND DISPLAY DATAFRAME
extracted_data = []

for letter_id, text in LETTERS.items():
    data = extract_fields(text)
    if data:
        data["letter_id"] = letter_id
        extracted_data.append(data)

df = pd.DataFrame(extracted_data)
cols = ["letter_id", "applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"]
df = df[cols]

df

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN
2,L003,Efua Darko,15000,purchase industrial sewing machines and fabric...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,buy feed and new layers for poultry farm,1500.0,True,18.0
4,L005,None,30000,buy a bulk order of yarn,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**

1. Using one of the test letters as a few-shot example causes data leakage to the train set. The model would see the ground-truth target output for that specific letter inside its prompt context, allowing it to simply memorize the pre-formatted answer rather than actually demonstrating that its instruction-following and extraction logic work on unseen data. Using a distinct example ensures correct evaluation of the model's true generalized performance.

2. Without explicit instructions to return null, LLMs default to hallucinating plausible values to fill missing fields. For instance, if an applicant does not mention their monthly profit, an unconstrained model will guess a number based on typical income or contextual hints. Enforcing null keeps the extracted dataset factually accurate.

3. For Extraction, Temperature of 0.0: Data extraction is a deterministic task with a single correct output format and accurate target value. Setting temperature to 0 forces greedy decoding, causing the model to pick the highest-probability tokens every time. This yields consistent, reproducible JSON schema formatting and exact factual extraction.

For Creative Tasks: Creative tasks like brainstorming, fiction writing, or marketing copy benefit from high entropy and variance. A non-zero temperature allows lower-probability tokens to be selected, introducing the varied phrasing, and unexpected ideas necessary for creative output.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [17]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.



# BRIEF_PROMPT (Written with student-level phrasing)
BRIEF_PROMPT = """You are a credit analyst assistant working at a microfinance bank in Ghana.
Your job is to read a loan application letter and write a clean decision-support brief to help a loan officer make a good decision.

CRITICAL RULE:
You are just giving advice, NOT making the final call. Do NOT approve or reject the loan yourself. Your job is to highlight the pros and cons so the human loan officer can decide.

Please break your response into these 4 EXACT sections:

1. Strengths: What looks good about this applicant or business? (e.g. good sales, savings history, clear plan, collateral, or guarantor).
2. Risks & Concerns: What are the main red flags or worries? (e.g. high loan amount, no collateral, bad cash flow, risky business idea).
3. Missing Information: What important details or documents did the applicant leave out that we need to check?
4. Suggested Next Steps: What concrete steps should the loan officer take next? (e.g. schedule an interview, visit their shop, check guarantor documents).

Keep your points clear, honest, and direct. Only stick to facts mentioned in the letter—do not make things up!"""

# BRIEF GENERATION HELPER FUNCTION
def generate_decision_brief(letter_id, letter_text):
    user_prompt = f"Here is the loan application letter to analyze ({letter_id}):\n\n{letter_text}"

    brief = ask_llm(
        user_prompt=user_prompt,
        system_prompt=BRIEF_PROMPT,
        temperature=0.0
    )
    return brief

# RUN DECISION SUPPORT ENGINE ON ALL SIX LETTERS
briefs = {}

for letter_id, text in LETTERS.items():

    print(f"DECISION SUPPORT BRIEF FOR: {letter_id}")


    brief_output = generate_decision_brief(letter_id, text)
    briefs[letter_id] = brief_output

    print(brief_output)
    print("\n")

DECISION SUPPORT BRIEF FOR: L001
## Step 1: Strengths
The applicant, Akosua Mensah, has a long-standing business with 12 years of experience selling provisions at Makola Market. She has a stable profit of GHS 900 per month from her current stall. Additionally, she has demonstrated a good savings history with the susu scheme, saving GHS 2,500 over two years without missing any contributions. She also has a clear plan for the loan, intending to use it to buy a deep freezer and expand into frozen foods. Furthermore, she has a guarantor, her sister, who is a teacher, potentially providing an added layer of security for the loan.

## Step 2: Risks & Concerns
One of the main concerns is the loan amount of GHS 8,000, which is significant compared to her monthly profit of GHS 900. Although she proposes to repay GHS 450 monthly over 20 months, which seems manageable given her current profit, there is a risk that her business expansion into frozen foods might not generate enough additional incom

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**

1. L003, Efua Darko - Strong Application: The model correctly picked up on her solid points—a registered business, strong monthly profit of GHS 2,800, peak seasonal revenue of GHS 22,000, a fixed deposit at GCB for collateral, and a clear repayment schedule. It also rightly noted that the bank still needs to physically verify her GCB deposit certificate and sales records.

L006,Kofi - Weak Application: The model spotted the major red flags immediately—asking for a large sum of GHS 50,000 for three unstarted businesses, no business track record, no collateral or guarantor, and no clear monthly repayment plan. It correctly treated his claim of being "trustworthy" as a personal opinion rather than real financial security.

2. Practical Reason: The AI model only reads what's written in the letter. It can't check bank balances or visit a shop in person to see if it actually exists. Letting an AI make the final decision could easily lead to major losses from fraud or incomplete information.

Ethical Reason: A loan can change someone's life and livelihood. Automated AI decisions can end up biased or unfair, and applicants deserve a human being who is accountable for the final decision and can evaluate their real situation fairly.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** f9ec9a274c0ebb6dec472b71daa5024d14bc1ca1

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [8]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

### Part 4.2 — Reliability: is the system consistent?

In [9]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

### Part 4.3 — Hallucination probing

In [10]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.